<a href="https://colab.research.google.com/github/GustavoNachbar/churn-dataset-clusters-classify-tests/blob/main/random_forest_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pandas as pd

df = pd.read_csv("/content/Churn_Modelling.csv")
# Renomeia as colunas do inglês original para português
df = df.rename(columns={
    'RowNumber': 'NumeroLinha',
    'CustomerId': 'IdCliente',
    'Surname': 'Sobrenome',
    'CreditScore': 'PontuacaoCredito',
    'Geography': 'Geografia',
    'Gender': 'Genero',
    'Age': 'Idade',
    'Tenure': 'TempoRelacionamento',
    'Balance': 'Saldo',
    'NumOfProducts': 'NumeroProdutos',
    'HasCrCard': 'PossuiCartaoCredito',
    'IsActiveMember': 'MembroAtivo',
    'EstimatedSalary': 'SalarioEstimado',
    'Exited': 'Saiu'
})

variaveis_numericas = [
    'PontuacaoCredito', 'Idade', 'TempoRelacionamento',
    'Saldo', 'NumeroProdutos', 'SalarioEstimado'
]

# 70% treino, 30% temporário
df_treino, df_temp = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["Saiu"]
)

# 15% teste, 15% validação
df_teste, df_validacao = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify=df_temp["Saiu"]
)

# Cluster treinado SÓ com cancelados do df_treino (nunca olha teste/validação)
df_cancelados_treino = df_treino[df_treino["Saiu"] == 1][variaveis_numericas]

scaler_perfil = StandardScaler()
X_scaled_cancelados = scaler_perfil.fit_transform(df_cancelados_treino)

kmeans_k4 = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans_k4.fit(X_scaled_cancelados)

# Aplicado (predict) nas 3 bases separadamente, sem re-treinar em nenhuma delas
for dataframe in [df_treino, df_teste, df_validacao]:
    X_scaled = scaler_perfil.transform(dataframe[variaveis_numericas])
    dataframe['Cluster_k4'] = kmeans_k4.predict(X_scaled)
    dataframe['Distancia_Centroide_k4'] = kmeans_k4.transform(X_scaled).min(axis=1)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef

features = variaveis_numericas + ['Cluster_k4', 'Distancia_Centroide_k4']

X_train = df_treino[features]
y_train = df_treino['Saiu']
X_test = df_teste[features]
y_test = df_teste['Saiu']

param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

grid_search_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid_rf, cv=5, scoring='f1', n_jobs=-1
)
grid_search_rf.fit(X_train, y_train)

cv_resultados = pd.DataFrame(grid_search_rf.cv_results_)

top10 = cv_resultados.sort_values('mean_test_score', ascending=False).head(10)[
    ['params', 'mean_test_score', 'std_test_score', 'rank_test_score']
]
top10['algoritmo'] = 'Random_Forest'  # troque para o nome de cada classificador

print(top10)
top10.to_csv('top10_random.csv', index=False)  # troque o nome do arquivo em cada notebook

melhor_modelo = grid_search_rf.best_estimator_
y_pred = melhor_modelo.predict(X_test)
y_proba = melhor_modelo.predict_proba(X_test)[:, 1]

resultado_rf_k4 = {
    'melhores_parametros': grid_search_rf.best_params_,
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_proba),
     'mcc': matthews_corrcoef(y_test, y_pred)
}
pd.DataFrame([resultado_rf_k4])

                                               params  mean_test_score  \
36  {'max_depth': 10, 'max_features': 'log2', 'min...         0.484734   
37  {'max_depth': 10, 'max_features': 'log2', 'min...         0.482847   
38  {'max_depth': 10, 'max_features': 'log2', 'min...         0.481491   
46  {'max_depth': 10, 'max_features': 'log2', 'min...         0.480598   
40  {'max_depth': 10, 'max_features': 'log2', 'min...         0.479015   
41  {'max_depth': 10, 'max_features': 'log2', 'min...         0.478639   
47  {'max_depth': 10, 'max_features': 'log2', 'min...         0.477075   
71  {'max_depth': None, 'max_features': 'log2', 'm...         0.476413   
63  {'max_depth': None, 'max_features': 'log2', 'm...         0.476316   
65  {'max_depth': None, 'max_features': 'log2', 'm...         0.475933   

    std_test_score  rank_test_score      algoritmo  
36        0.024782                1  Random_Forest  
37        0.020924                2  Random_Forest  
38        0.026764        

,melhores_parametros,accuracy,precision,recall,f1,roc_auc,mcc
0,"{'max_depth': 10, 'max_features': 'log2', 'min...",0.844667,0.718563,0.392157,0.5074,0.841104,0.45197
